In [2]:
# source: https://www.kaggle.com/discussions/general/74235
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

# MBTI Datasets
! kaggle datasets download -d "zeyadkhalid/mbti-personality-types-500-dataset" # cleaned personalitycafe + reddit
! kaggle datasets download -d "jhonatanparada/cleaned-mbti-personality-type-twitter-dataset" # cleaned twitter

! unzip "*.zip"

Dataset URL: https://www.kaggle.com/datasets/zeyadkhalid/mbti-personality-types-500-dataset
License(s): CC0-1.0
100% 123M/123M [00:01<00:00, 79.0MB/s]

Dataset URL: https://www.kaggle.com/datasets/jhonatanparada/cleaned-mbti-personality-type-twitter-dataset
License(s): unknown
100% 7.15M/7.15M [00:00<00:00, 49.0MB/s]

Archive:  mbti-personality-types-500-dataset.zip
  inflating: MBTI 500.csv            

Archive:  cleaned-mbti-personality-type-twitter-dataset.zip
  inflating: cleaned_twitter_MBTI.csv  

2 archives were successfully processed.


In [28]:
import pandas as pd
import numpy as np

# Matplotlib, plt and cm modules
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# the Naive Bayes model
from sklearn.naive_bayes import MultinomialNB                # Classifiers
from sklearn.linear_model import SGDClassifier

# Testing
from sklearn.model_selection import train_test_split

# function for transforming documents into counts
from sklearn.feature_extraction.text import CountVectorizer  # Vectorizer
from sklearn.feature_extraction.text import TfidfTransformer # Transformer
from sklearn.feature_extraction.text import TfidfVectorizer # Transformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import LabelEncoder # function for encoding categories

from sklearn import metrics

DATASETS_PATH = './'

In [7]:
cleaned_reddit_percafe = pd.read_csv(f'{DATASETS_PATH}MBTI 500.csv')
cleaned_twitter = pd.read_csv(f'{DATASETS_PATH}cleaned_twitter_MBTI.csv')

In [11]:
print(cleaned_twitter.isnull().sum())

cleaned_twitter

label                    0
ie                       0
ns                       0
tf                       0
jp                       0
lemmatize_clean_posts    0
dtype: int64


,label,ie,ns,tf,jp,lemmatize_clean_posts
0,INTJ,I,N,T,J,the pope be infallible this be a catholic dogm...
1,INTJ,I,N,T,J,be you make you look cute on because then I ca...
2,INTJ,I,N,T,J,les balle sont relle et sont tire tr rapidemen...
3,INTJ,I,N,T,J,I m like entp but idiotic hey boy do you want ...
4,INTJ,I,N,T,J,give it to he have pica since childhood say qu...
...,...,...,...,...,...,...
7806,INTP,I,N,T,P,godpls take care hiro emergency room be you ok...
7807,INTP,I,N,T,P,wow last time I get intp I think u upset the f...
7808,ENTP,E,N,T,P,a that someone will get his ass kick so its ok...
7809,INFJ,I,N,F,J,if you re intj this one be for you what be nev...


In [12]:
print(cleaned_twitter.isnull().sum())

cleaned_reddit_percafe

label                    0
ie                       0
ns                       0
tf                       0
jp                       0
lemmatize_clean_posts    0
dtype: int64


,posts,type
0,know intj tool use interaction people excuse a...,INTJ
1,rap music ehh opp yeah know valid well know fa...,INTJ
2,preferably p hd low except wew lad video p min...,INTJ
3,drink like wish could drink red wine give head...,INTJ
4,space program ah bad deal meing freelance max ...,INTJ
...,...,...
106062,stay frustrate world life want take long nap w...,INFP
106063,fizzle around time mention sure mistake thing ...,INFP
106064,schedule modify hey w intp strong wing underst...,INFP
106065,enfj since january busy schedule able spend li...,INFP


In [13]:
# source: https://www.kaggle.com/code/rajshreev/mbti-personality-predictor-using-machine-learning
def get_types(row):
    t=row['type'] # or t

#    I = 0; N = 0
#    T = 0; J = 0

    if t[0] == 'I': I = 'I'
    elif t[0] == 'E': I = 'E'
    else: print('I-E not found')

    if t[1] == 'N': N = 'N'
    elif t[1] == 'S': N = 'S'
    else: print('N-S not found')

    if t[2] == 'T': T = 'T'
    elif t[2] == 'F': T = 'F'
    else: print('T-F not found')

    if t[3] == 'J': J = 'J'
    elif t[3] == 'P': J = 'P'
    else: print('J-P not found')
    return pd.Series( {'ie':I, 'ns':N , 'tf': T, 'jp': J })



## Add Binary columns for MBTI dimensions

In [14]:
cleaned_reddit_percafe = cleaned_reddit_percafe.join(cleaned_reddit_percafe.apply(
    lambda row: get_types (row),axis=1))

cleaned_reddit_percafe.head(5)


,posts,type,ie,ns,tf,jp
0,know intj tool use interaction people excuse a...,INTJ,I,N,T,J
1,rap music ehh opp yeah know valid well know fa...,INTJ,I,N,T,J
2,preferably p hd low except wew lad video p min...,INTJ,I,N,T,J
3,drink like wish could drink red wine give head...,INTJ,I,N,T,J
4,space program ah bad deal meing freelance max ...,INTJ,I,N,T,J


In [ ]:
# if wanting to export
# data.to_csv("cleaned_twitter_MBTI.csv", index=False)

In [16]:
# Concatenate or perform union for both datasets

cleaned_twitter = cleaned_twitter.rename(
    columns={'lemmatize_clean_posts': 'posts', 'label': 'type'})

all_datasets = pd.concat([cleaned_reddit_percafe, cleaned_twitter], ignore_index=True)
all_datasets

,posts,type,ie,ns,tf,jp
0,know intj tool use interaction people excuse a...,INTJ,I,N,T,J
1,rap music ehh opp yeah know valid well know fa...,INTJ,I,N,T,J
2,preferably p hd low except wew lad video p min...,INTJ,I,N,T,J
3,drink like wish could drink red wine give head...,INTJ,I,N,T,J
4,space program ah bad deal meing freelance max ...,INTJ,I,N,T,J
...,...,...,...,...,...,...
113873,godpls take care hiro emergency room be you ok...,INTP,I,N,T,P
113874,wow last time I get intp I think u upset the f...,INTP,I,N,T,P
113875,a that someone will get his ass kick so its ok...,ENTP,E,N,T,P
113876,if you re intj this one be for you what be nev...,INFJ,I,N,F,J


## Binary Classifiers
### naïve Bayes classifier (MultinomialNB)

In [30]:
MBTI_subtype = ['ie', 'ns', 'tf', 'jp']
binary_models = {}

for subtype in MBTI_subtype:

  print(all_datasets[subtype].value_counts())

  encoder = LabelEncoder()
  y = encoder.fit_transform(all_datasets[subtype])

  # split into train and test sets
  x_train, x_test, y_train, y_test = train_test_split(all_datasets['posts'], y, test_size=0.3)

  text_clf = Pipeline([
      ('vect', CountVectorizer()),
      ('tfidf', TfidfTransformer()),
      ('clf', MultinomialNB()),
      ])

  text_clf.fit(x_train, y_train)

  binary_models[subtype] = text_clf

  predicted = text_clf.predict(x_test)
  print(np.mean(predicted == y_test))

  #print(
  #    f"Classification report for Kaggle dataset:\n"
  #    f"{metrics.classification_report(y_test, predicted, target_names=encoder.classes_)}\n"
  #)

ie
I    85925
E    27953
Name: count, dtype: int64
0.756468797564688
ns
N    102900
S     10978
Name: count, dtype: int64
0.9054267650158061
tf
T    72418
F    41460
Name: count, dtype: int64
0.6492214026460602
jp
P    65999
J    47879
Name: count, dtype: int64
0.5857627912422433


## support vector machine (SVM) (SDGClassifier)

In [31]:
# Source:
MBTI_subtype = ['ie', 'ns', 'tf', 'jp']
binary_models = {}

for subtype in MBTI_subtype:

  print(all_datasets[subtype].value_counts())

  encoder = LabelEncoder()
  y = encoder.fit_transform(all_datasets[subtype])

  # split into train and test sets
  x_train, x_test, y_train, y_test = train_test_split(all_datasets['posts'], y, test_size=0.3)

  text_clf = Pipeline([
      ('vect', CountVectorizer()),
      ('tfidf', TfidfTransformer()),
      ('clf',SGDClassifier(loss='hinge', penalty='l2',
                           alpha=1e-3, random_state=42,
                           max_iter=5, tol=None)),
      ])

  text_clf.fit(x_train, y_train)
  binary_models[subtype] = text_clf

  predicted = text_clf.predict(x_test)
  print(np.mean(predicted == y_test))

  #print(
  #    f"Classification report for Kaggle dataset:\n"
  #    f"{metrics.classification_report(y_test, predicted, target_names=encoder.classes_)}\n"
  #)

ie
I    85925
E    27953
Name: count, dtype: int64
0.7590446083596768
ns
N    102900
S     10978
Name: count, dtype: int64
0.9029680365296804
tf
T    72418
F    41460
Name: count, dtype: int64
0.86125746399719
jp
P    65999
J    47879
Name: count, dtype: int64
0.7404577918276548
